# Predicting Stellar Class — Preprocessing & EDA

Kaggle Playground Series S6E6. Multiclass classification: `GALAXY` / `STAR` / `QSO`.
Metric — **balanced accuracy** (mean per-class recall), so class imbalance matters.

Notebook goals:
1. Load and validate the data (shapes, dtypes, missing values, duplicates, train/test consistency).
2. Full EDA: target variable, numeric and categorical features, relation to the class, correlations, anomalies.
3. Build a clean feature set for modeling and persist it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PURPLE_BLUE = ['#3B2F8F', '#6A5ACD', '#9D7BE0', '#4169E1', '#7B68EE', '#5B8DEF']
CLASS_PALETTE = {'GALAXY': '#3B2F8F', 'QSO': '#6A5ACD', 'STAR': '#9D7BE0'}
SEQ_CMAP = 'BuPu'

sns.set_theme(style='whitegrid', palette=PURPLE_BLUE)
plt.rcParams['figure.dpi'] = 110

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    train_paths = sorted(Path('/kaggle/input').rglob('train.csv'))
    DATA_DIR = train_paths[0].parent
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../docs/dataset')
    OUT_DIR = Path('../data_processed')
print('Environment:', 'Kaggle' if ON_KAGGLE else 'local')
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)

TARGET = 'class'
ID = 'id'
CLASSES = ['GALAXY', 'QSO', 'STAR']


def reduce_mem(df):
    '''
    Downcast numeric columns (float64 -> float32, int64 -> smallest int)
    to roughly halve memory usage, which matters under Kaggle RAM limits.
    '''
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

## 1. Data loading

In [ ]:
train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train:', train.shape)
print('test :', test.shape)
print('sub  :', sample_sub.shape)
print('train memory: %.1f MB' % (train.memory_usage(deep=True).sum() / 1024**2))
train.head()

**Observations.** Train has ~577k rows and 12 columns; test has ~247k rows and 11 columns (no `class`). The feature schema matches between train and test, and `sample_submission` aligns with the test size, so the split is well formed.

## 2. Basic data quality checks

In [ ]:
train.info(show_counts=True)

**Observations.** 8 numeric columns (`float64`), 1 integer `id`, and 3 `object` columns (`spectral_type`, `galaxy_population`, `class`). Every column is fully populated (577,347 non-null), confirming no missing data in train at the dtype level.

In [ ]:
miss = pd.DataFrame({
    'train_nulls': train.isnull().sum(),
    'test_nulls': test.reindex(columns=train.columns).isnull().sum(),
})
print(miss)
print('\nTotal nulls train:', train.isnull().sum().sum(), '| test:', test.isnull().sum().sum())

**Observations.** No missing values anywhere except `class` in test, which is expected since the target is what we predict. No imputation strategy is required.

In [ ]:
print('Duplicate ids in train:', train[ID].duplicated().sum())
print('Duplicate ids in test :', test[ID].duplicated().sum())
print('Shared ids train/test:', len(set(train[ID]) & set(test[ID])))

train_feats = set(train.columns) - {TARGET}
test_feats = set(test.columns)
print('\nColumns only in train (excluding class):', train_feats - test_feats)
print('Columns only in test :', test_feats - train_feats)

feat_cols = [c for c in train.columns if c not in (ID, TARGET)]
print('\nFully duplicated feature rows (train):', train.duplicated(subset=feat_cols).sum())

**Observations.** No duplicate `id` in either split, no overlap of ids between train and test, and no fully duplicated feature rows. The feature columns are identical across splits (only `class` is train-exclusive), so the data is consistent and leak-free.

In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
cat_cols = ['spectral_type', 'galaxy_population']
print('Numeric    :', num_cols)
print('Categorical:', cat_cols)
train[num_cols].describe().T

**Observations.** The five photometric magnitudes (`u, g, r, i, z`) sit in a similar 12–28 range, hinting at strong correlation among them. `redshift` is heavily right-skewed (mean 0.72, max 7.0) with a small negative minimum. `alpha` spans the full 0–360° RA range and `delta` the declination range, as expected for sky coordinates.

## 3. Target variable

In [ ]:
vc = train[TARGET].value_counts()
vc_norm = train[TARGET].value_counts(normalize=True)
print(pd.DataFrame({'count': vc, 'share': vc_norm.round(4)}))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=vc.index, y=vc.values, ax=ax[0],
            hue=vc.index, legend=False,
            palette=[CLASS_PALETTE.get(c, '#6A5ACD') for c in vc.index])
ax[0].set_title('Count by class')
ax[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', startangle=90,
          colors=[CLASS_PALETTE.get(c, '#6A5ACD') for c in vc.index])
ax[1].set_title('Class shares')
plt.tight_layout(); plt.show()

**Observations.** The target is clearly imbalanced: GALAXY dominates (~65%), QSO is mid-sized (~20%), and STAR is the minority (~14%). Because the metric is balanced accuracy, the minority STAR class must be protected — use `StratifiedKFold` and `balanced_accuracy_score`, and consider class weighting.

## 4. Numeric features: distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(train[col], bins=80, ax=ax, color='#6A5ACD')
    ax.set_title(col)
plt.suptitle('Numeric feature distributions (train)', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The magnitudes are roughly bell-shaped with mild left skew. `redshift` is strongly right-skewed with a spike near zero (stars) and a long tail (quasars) — this multimodality already foreshadows its discriminative power. `alpha`/`delta` are broad and fairly flat, typical of survey footprint coverage.

In [ ]:
for col in num_cols:
    neg = (train[col] < 0).sum()
    print(f'{col:9s}  min={train[col].min():10.4f}  max={train[col].max():10.4f}  <0: {neg}')

**Observations.** Negative `delta` values (~103k) are just southern-hemisphere declinations and are valid. Negative `redshift` (~9k) is physically plausible (peculiar velocities / measurement noise) and should be kept. Only a single negative `u` magnitude appears — a likely outlier, but not worth special handling given its rarity.

## 5. Numeric features by class

In [ ]:
print(train.groupby(TARGET)['redshift'].describe().round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for cls in CLASSES:
    sns.kdeplot(train.loc[train[TARGET] == cls, 'redshift'].clip(-0.5, 4),
                label=cls, ax=ax[0], fill=True, alpha=0.3, color=CLASS_PALETTE[cls])
ax[0].set_title('redshift by class (clip -0.5..4)'); ax[0].legend()
sns.boxplot(data=train, x=TARGET, y='redshift', order=CLASSES, ax=ax[1],
            hue=TARGET, legend=False,
            palette=[CLASS_PALETTE[c] for c in CLASSES])
ax[1].set_ylim(-0.5, 4); ax[1].set_title('redshift boxplot by class')
plt.tight_layout(); plt.show()

**Observations.** `redshift` separates the classes almost perfectly: STAR clusters at ≈ 0, GALAXY occupies a moderate range, and QSO sits at high redshift. This is the single most powerful predictor and should drive most of the model's accuracy.

In [ ]:
other_num = [c for c in num_cols if c != 'redshift']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), other_num):
    sns.boxplot(data=train, x=TARGET, y=col, order=CLASSES, ax=ax, showfliers=False,
                hue=TARGET, legend=False,
                palette=[CLASS_PALETTE[c] for c in CLASSES])
    ax.set_title(col)
for ax in axes.ravel()[len(other_num):]:
    ax.axis('off')
plt.suptitle('Numeric features by class (plot outliers hidden)', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The raw magnitudes overlap heavily across classes — individually they are weak separators. The class medians differ only modestly, which reinforces that derived colors (band differences) should be more informative than the magnitudes themselves.

## 6. Categorical features

In [ ]:
for col in cat_cols:
    print(f'=== {col} ===')
    print(train[col].value_counts())
    print('unique in test but missing from train:', set(test[col].unique()) - set(train[col].unique()))
    print()

**Observations.** `spectral_type` has 4 levels (M most common, O/B rarest) and `galaxy_population` is binary. No unseen categories appear in test, so a simple ordinal/category encoding is safe with no fallback needed for unknown values.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, col in zip(ax, cat_cols):
    ct = pd.crosstab(train[col], train[TARGET], normalize='index')[CLASSES]
    ct.plot(kind='bar', stacked=True, ax=a, color=[CLASS_PALETTE[c] for c in CLASSES])
    a.set_title(f'Class share within {col}'); a.set_ylabel('share')
    a.legend(title='class', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout(); plt.show()

print(pd.crosstab(train['spectral_type'], train[TARGET], normalize='index').round(3))

**Observations.** Class composition shifts strongly with `spectral_type`: the O/B (hot) type is dominated by QSO, while cooler types lean toward GALAXY/STAR. `galaxy_population` also splits the classes meaningfully. Both categoricals carry real signal and should be kept as features.

## 7. Correlations and spatial distribution

In [ ]:
corr = train[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap=SEQ_CMAP, square=True)
plt.title('Numeric feature correlations')
plt.tight_layout(); plt.show()

**Observations.** The `u, g, r, i, z` bands are very highly correlated with one another (multicollinearity), confirming the earlier hypothesis. Color differences (`u-g`, `g-r`, …) decorrelate this block and are expected to be more informative than the raw magnitudes.

In [ ]:
samp = train.sample(40000, random_state=42)
plt.figure(figsize=(11, 5))
sns.scatterplot(data=samp, x='alpha', y='delta', hue=TARGET, hue_order=CLASSES,
                palette=CLASS_PALETTE, s=6, alpha=0.4, linewidth=0)
plt.title('Object positions on the sky (alpha vs delta), 40k sample')
plt.tight_layout(); plt.show()

**Observations.** Classes are well mixed across the sky with no obvious spatial clustering by class. The `alpha`/`delta` coordinates therefore carry little direct class signal and are unlikely to be strong predictors on their own.

## 8. train vs test comparison (drift)
The dataset is synthetic — let's confirm that feature distributions are consistent between train and test.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.kdeplot(train[col].clip(train[col].quantile(.001), train[col].quantile(.999)),
                label='train', ax=ax, color='#3B2F8F')
    sns.kdeplot(test[col].clip(test[col].quantile(.001), test[col].quantile(.999)),
                label='test', ax=ax, color='#9D7BE0')
    ax.set_title(col); ax.legend()
plt.suptitle('train vs test: feature distributions', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The train and test distributions overlap almost perfectly for every numeric feature — no meaningful covariate shift. A model validated on train CV should generalize reliably to the test set without drift-correction tricks.

## 9. Preprocessing and feature engineering
- `id` is not used as a feature.
- Build color indices from the photometry (physically meaningful SDSS features).
- Encode categorical features (ordered codes; for gradient boosting they can be passed as `category`).
- Encode the target into integer labels.

In [ ]:
def add_features(df):
    '''
    Add SDSS color indices: differences between adjacent photometric bands.
    These colors are the standard, physically meaningful features for SDSS
    object classification and carry more signal than raw magnitudes.
    '''
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_z'] = df['u'] - df['z']
    return df

train_fe = add_features(train)
test_fe = add_features(test)
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'u_z']

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, color_cols):
    sns.boxplot(data=train_fe, x=TARGET, y=col, order=CLASSES, ax=ax, showfliers=False,
                hue=TARGET, legend=False,
                palette=[CLASS_PALETTE[c] for c in CLASSES])
    ax.set_title(col)
plt.tight_layout(); plt.show()

**Observations.** The color indices show clearer class separation than the raw bands — `u_g` and `u_z` in particular shift noticeably between classes. This validates adding the colors as engineered features.

In [ ]:
spectral_order = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
pop_map = {'Blue_Cloud': 0, 'Red_Sequence': 1}

for df in (train_fe, test_fe):
    df['spectral_type_code'] = df['spectral_type'].map(spectral_order)
    df['galaxy_population_code'] = df['galaxy_population'].map(pop_map)

class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
train_fe['target'] = train_fe[TARGET].map(class_to_int)

feature_cols = num_cols + color_cols + ['spectral_type_code', 'galaxy_population_code']
print('Final features (%d):' % len(feature_cols))
print(feature_cols)
print('\nNull check after encoding:',
      train_fe[feature_cols].isnull().sum().sum(), test_fe[feature_cols].isnull().sum().sum())

**Observations.** 15 final features (8 numeric + 5 colors + 2 encoded categoricals) with zero nulls in both train and test after encoding. The ordinal `spectral_type` mapping (hot → cool) preserves a physically meaningful ordering. The feature matrix is model-ready.

## 10. Persisting the feature set
Save the engineered `train`/`test` matrices so the downstream modeling notebook can load them directly without repeating preprocessing.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_fe[[ID] + feature_cols + ['target', TARGET]].to_parquet(OUT_DIR / 'train_fe.parquet', index=False)
test_fe[[ID] + feature_cols].to_parquet(OUT_DIR / 'test_fe.parquet', index=False)

print('Saved to', OUT_DIR.resolve())
print('train_fe:', train_fe[[ID] + feature_cols + ['target']].shape, '| test_fe:', test_fe[[ID] + feature_cols].shape)

**Observations.** The engineered feature set is persisted to `OUT_DIR` (`/kaggle/working` on Kaggle, a local folder otherwise) as `train_fe.parquet` (features + `target` + `class`) and `test_fe.parquet` (features only). Modeling and the actual `submission.csv` are intentionally left to a separate notebook — this one stays a pure EDA + feature-prep step.

## EDA conclusions

**Data quality.** The data is clean and model-ready: zero missing values (except the target in test), no duplicate `id`, no fully duplicated rows, and identical feature schemas across train/test. Negative `delta` (~104k rows) and negative `redshift` (~9k rows) are physically valid and should be kept as-is.

**Target.** Strongly imbalanced — GALAXY 65.4% / QSO 20.3% / STAR 14.3%. Since the metric is **balanced accuracy** (mean per-class recall), the minority **STAR** class dominates the score: a model that is excellent on GALAXY/QSO but mediocre on STAR will score poorly.

**Signal structure.** Almost all class signal is concentrated in two features:
- **`redshift`** — the single strongest separator. Class means: STAR ≈ 0.07, GALAXY ≈ 0.51, QSO ≈ 1.88. STAR clusters tightly near zero, QSO sits high, GALAXY in between.
- **`spectral_type`** — near-deterministic for some levels: `M → 95% GALAXY`, `O/B → 71% QSO`, `A/F → 50% QSO`, `G/K → 57% GALAXY`.

**Weaker signal.** Raw `u, g, r, i, z` magnitudes are heavily mutually correlated and overlap across classes; the derived **color indices** (`u_g`, `u_z`, …) separate classes better. Sky coordinates `alpha`/`delta` carry essentially no class signal.

**No drift.** train and test distributions overlap almost perfectly across every numeric feature, so CV scores should track the leaderboard closely — no adversarial-validation / drift correction needed.

## Key findings & hypotheses

A few data-driven takeaways that should shape the modeling strategy:

- **The score battle is the STAR class.** With shares 65/20/14, balanced accuracy is bottlenecked by STAR. STAR lives at `redshift ≈ 0`, which **overlaps with the low-redshift tail of GALAXY** — this boundary is where most confusion is expected, so STAR↔GALAXY separation deserves dedicated features.
- **Suspicious "stars" at high redshift.** STAR `redshift` reaches a max of ≈ **5.4**, which is physically impossible for a real star (stars sit at ≈ 0). These are either synthetic noise or mislabeled QSO-like objects. They are a likely source of error and a candidate for an explicit anomaly flag.
- **`spectral_type` is almost a rule-based classifier.** `M → 95% GALAXY` and `O/B → 71% QSO` mean a model can lean heavily on it; combined with `redshift` it should resolve most of the easy cases.
- **Redundant photometry.** The five bands are near-collinear, so their raw values add little beyond the colors — dimensionality can be trimmed with no loss.

## Recommendations — feature engineering

- **Keep the color indices** — they already separate classes better than raw magnitudes. Consider extending to **all pairwise colors** (`u-r`, `g-i`, `r-z`, …), then prune by importance.
- **Tame `redshift` skew** — it spans 0–7 with a heavy right tail. Add `log1p(redshift)` and/or coarse redshift **bins**; tree models won't need it but it helps linear models / NNs and makes interactions cleaner.
- **Anomaly flags** — engineer `is_neg_redshift` and `is_star_like_z` (`redshift` close to 0). The high-redshift STAR tail (max ≈ 5.4, physically impossible for a star) is a prime suspect for misclassification — an explicit flag may help the model isolate it.
- **Interactions** — `redshift × spectral_type` is the most promising, since both are top signals and their combination is near-deterministic for several cells of the crosstab.
- **Target / categorical encoding** — `spectral_type` is so predictive that **target encoding** (done inside CV folds to avoid leakage) or CatBoost's native categorical handling is likely to beat the plain ordinal code.
- **Drop weak features** — `alpha`/`delta` show no class signal; keep them only if they help OOF, otherwise drop to reduce noise.

## Recommendations — modeling & validation

- **Model family.** Prefer gradient boosting (**LightGBM / XGBoost / CatBoost**) over ExtraTrees — stronger on tabular data and faster on 577k rows. **CatBoost** is attractive because it handles `spectral_type` / `galaxy_population` natively as categoricals.
- **Validation.** `StratifiedKFold` (5 folds) to preserve the class ratio, and score with `balanced_accuracy_score`. There is no drift, so OOF should track the leaderboard well.
- **Imbalance handling.** Because the metric rewards per-class recall, use `class_weight='balanced'` / `sample_weight`, or boosting's `is_unbalance` / `scale_pos_weight` analogues. Do **not** simply optimize plain accuracy.
- **Focus on STAR.** STAR is the expected bottleneck. Track the per-class recall and the STAR row of the confusion matrix every iteration — that single number drives balanced accuracy more than overall accuracy does.
- **Calibration / thresholds.** For balanced accuracy, tuning per-class decision thresholds on OOF probabilities can beat the default `argmax`.

## Recommendations — pushing the score further

- **External / original data.** Playground S6E6 is synthesized from the real *SDSS Stellar Classification* dataset. Concatenating the original data into the training set is a classic Playground-series tactic and frequently the single biggest score jump — worth trying early.
- **Pseudo-labeling.** Given the strong, low-noise signal, high-confidence test predictions can be folded back into training as pseudo-labels for a small extra boost.
- **Ensembling / blending.** Blend a GBDT (LightGBM/CatBoost) with a different model family (e.g. a small MLP on standardized colors + redshift). The decorrelated errors usually lift balanced accuracy.
- **Per-class threshold tuning.** Instead of `argmax`, optimize per-class decision thresholds on OOF predictions directly for balanced accuracy — cheap and targeted at the STAR/GALAXY boundary.

## Suggested roadmap

A pragmatic order of work for the modeling notebook, highest expected return first:

1. **Strong single model** — LightGBM/CatBoost on the 15 features, `StratifiedKFold(5)`, optimizing balanced accuracy with class weights. Establishes a real reference score.
2. **Attack STAR recall** — add the redshift-anomaly features, inspect the STAR row of the confusion matrix, tune class weights / decision thresholds specifically for STAR.
3. **Feature expansion** — full pairwise colors + target-encoded `spectral_type`; keep only what improves OOF.
4. **External SDSS data** — append the original dataset to train; often the single biggest jump in Playground series.
5. **Ensembling** — blend LightGBM + CatBoost + XGBoost (and optionally a NN) on OOF predictions.

> This notebook deliberately stops at clean EDA + a persisted feature set. All modeling, validation, and submission generation live in a separate notebook that loads `train_fe.parquet` / `test_fe.parquet`.